# Local benchmark with Ollama

The notebook uses the same functions as the script: one call per question, merge
by `id_prova` and `Nquestao` with `validate="one_to_one"`, and incremental saving.
The warmup is kept apart from the questions and the metrics. Run it from the
repository root or from `notebooks`.

Old outputs were removed. Running the benchmark cell below calls the local
Ollama and may download the model if it is missing.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "teste_modelo_aut.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.teste_modelo_aut import (
    load_input, run_model, load_progress,
    build_config, merge_results, print_report, output_path,
)


In [ ]:
file = "provas_tjdft_com_gabarito"
model = "gemma4:e4b-it-q4_K_M"
output_dir = ROOT / "resultados" / "notebook"
resume = False  # To continue the same run, explicitly change this to True.
path, exam_2015 = load_input(file, ROOT / "data" / "processed")
exam_2015.head()


In [ ]:
run_model(
    models=(model,), files=(file,), resume=resume,
    csv_dir=ROOT / "data" / "processed", output_dir=output_dir,
)


The cell below can also be used after an interruption, without new inferences.
Each saved answer contains `modelo`, `arquivo`, `id_prova` and `Nquestao`.
Only processed questions with an A–E answer key count towards accuracy; annulled
questions and those without an answer key are kept but excluded from the
evaluation. Invalid answers and inference failures on evaluable questions count
as incorrect. The report flags partial runs. The count of answers without a
letter includes inference failures, so these two counts may overlap.
`carregamento` describes CPU/GPU allocation, not compute utilization.
`possivel_truncamento` is only a heuristic and stays unknown without a token
count. The progress JSON is the source for resuming; do not edit it by hand.


In [ ]:
progress = output_path(model, file, output_dir).with_suffix(".progresso.json")
results = load_progress(
    progress, build_config(model, file, path, exam_2015), exam_2015,
)
final_df = merge_results(exam_2015, results)
print_report(final_df, model, file)
final_df.head()
